# Husky 와이어태핑(Wire-Tapping)을 통한 오류주입 공격(FIA) Trace 수집

## 두 ChipWhisperer 장치의 역할 분리 실험 — Lite (통신·프로그래머·글리치 미발생) + Husky (수동 관측자 + 전압 글리치 발생기)

---

### 🎯 노트북의 목표

본 노트북은 ChipWhisperer에 처음 입문하는 동료 연구자를 대상으로, **두 대의 ChipWhisperer 장치를 동시에 운용하여 와이어태핑(wire-tapping) 방식으로 오류주입 공격(FIA, Fault Injection Attack) 실험** 을 수행하는 절차를 정리한 자료입니다.
즉, **부채널 측정용 와이어태핑 셋업**과 **전압 글리치 기반 오류주입**을 결합해, 제3자가 일부 신호선과 전원선에 물리적으로 접근할 수 있다는 공격 가정을 실험실 장비의 역할 분리로 모형화합니다. 실제 제품 환경의 접근 난이도나 공격 가능성을 입증하는 실험은 아닙니다.

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 라이브러리 임포트 및 다중 장치(Lite + Husky) 동시 연결 | `lite_scope`, `husky_scope` |
| **2단계** | Lite ↔ 타겟 보드 통신 채널 확보 (SimpleSerial2) | `target` 객체 |
| **3단계** | 펌웨어 빌드 + Lite를 경유한 타겟 보드 프로그래밍 | 플래싱 완료된 타겟 |
| **4단계** | 골든 모델(Golden Model)을 이용한 통신·연산 정상성 검증 | `Golden_k_XOR_p` |
| **5단계** | **Husky 와이어태핑 + 전압 글리치 환경 구성** (외부 클럭 동기화 + 트리거 + 글리치 모듈) | 측정·글리치 동시 준비된 Husky |
| **6단계** | `Encrypt()` + **글리치 파라미터 스윕** + 인터랙티브 실시간 모니터링 | `t_husky`, `glitch_parameter`, `i_k/i_p/o_c` |
| **7단계** | 자원 해제 (USB / UART / 메모리) | 다음 세션 충돌 방지 |

> 본 자료는 SCA 와이어태핑 입문에 이어지는 **응용편(FIA + Wire-Tapping)** 성격의 연구 노트입니다.
> SimpleSerial 패킷 구조, `my_fsr_cmd()` 헬퍼, 다중 장치 연결 등의 기본기는 본 노트북 내에서 다시 정리하되, **와이어태핑 환경에서 전압 글리치를 안정적으로 주입·관측하는 절차** 를 집중적으로 다룹니다.


---

## 🔍 시작하기 전에

### 와이어태핑(Wire-Tapping) + 오류주입(FIA) 결합 시나리오

전통적인 ChipWhisperer 실습에서는 단일 보드가 **타겟에게 평문을 보내고 → 연산을 트리거하고 → 전력 파형을 측정** 하는 동시에 **글리치를 주입** 합니다.
즉 공격자가 통신·전원·클럭 라인을 모두 직접 제어할 수 있다는 가정입니다.

그러나 실제 오류주입 공격 환경은 그렇게 협조적이지 않습니다.
- 공격자는 **이미 동작 중인 시스템의 신호선에 측정용 프로브와 글리치 주입용 프로브만 부착** 할 수 있을 뿐, 정상 통신 흐름에는 개입할 수 없습니다.
- 따라서 측정·글리치 장비는 **외부에서 신호선을 보고**, **외부에서 전원선에 펄스를 가하는** 수동적+능동적 혼합 역할을 수행합니다.

본 노트북은 이러한 **현실적 FIA 시나리오** 를 두 대의 ChipWhisperer 장치 분담으로 재현합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│  공격 시나리오 모형                                                  │
│                                                                     │
│   ChipWhisperer-Lite   ───  "정상 사용자" 역할                       │
│      └─ 타겟 보드와 UART (SimpleSerial2) 로 통신                     │
│      └─ 타겟 펌웨어를 컴파일·플래싱                                   │
│      └─ 타겟의 시스템 클럭 공급(HS2)                                 │
│      └─ 글리치 / 측정 에는 일체 관여하지 않음                        │
│                                                                     │
│   ChipWhisperer-Husky  ───  "은밀한 관측자 + 공격자" 역할            │
│      └─ 트리거 / 클럭 / 전력 라인을 분기 받아 측정 (passive)         │
│      └─ 전압 글리치 펄스를 타겟 VCC 라인에 주입 (active)             │
│      └─ 통신에는 일체 개입하지 않음                                  │
└─────────────────────────────────────────────────────────────────────┘
```

### 단일 칩위스퍼러 FIA 환경과 본 노트북 환경의 차이

| 항목 | 단일 장치 FIA | 본 노트북 (다중 장치 와이어태핑 FIA) |
|:----:|:----:|:----:|
| 측정 / 글리치 장비 수 | 1대 (Husky)            | **2대 (Lite + Husky)** |
| UART 통신 주체        | Husky                   | **Lite** 전담 |
| 펌웨어 프로그래머      | Husky                   | **Lite** 전담 |
| 타겟 클럭 공급원       | Husky 의 `clkgen`        | **Lite 의 `clkgen`** (HS2) |
| Husky 의 역할         | 통신 + 측정 + 글리치    | **측정 + 글리치만** (passive observer + active fault injector) |
| Husky 클럭 소스        | 내부 PLL (`clkgen`)     | **외부 클럭(`extclk_aux_io`)** + 주파수 탐색 |
| 현실성 (공격 시나리오) | 낮음 (자기 자신을 측정·공격) | **높음** (제3자가 신호선 도청 + 전원 외란) |


### 본 노트북이 사용하는 글리치 종류 — **전압 글리치(Voltage Glitch)**

```
┌─────────────────────────────────────────────────────────────┐
│   1) 클럭 글리치 (Clock Glitch)                              │
│      → 클럭 신호에 짧은 추가 펄스 삽입 → 셋업 타임 위반      │
│      → 타겟의 시스템 클럭 라인을 공격자가 제어해야 함        │
│                                                             │
│   2) 전압 글리치 (Voltage Glitch)   ← 본 노트북의 주제       │
│      → VCC 라인을 순간적으로 떨어뜨림 → 회로 오동작          │
│      → 타겟의 전원 라인만 분기하면 됨 (와이어태핑과 잘 어울림)│
│                                                             │
│   3) 광학/EM 펄스, 레이저(FIB), 온도 등                      │
│      → 본 ChipWhisperer 플랫폼 범위 외                       │
└─────────────────────────────────────────────────────────────┘
```

> 💡 **왜 와이어태핑 환경에서 전압 글리치인가?**
> 클럭 글리치는 공격자가 타겟의 시스템 클럭 라인을 *교체* 해야 가능합니다.
> 그러나 와이어태핑 시나리오에서는 클럭은 **이미 Lite 가 공급 중** 이며, Husky 는 그 클럭을 *관측만* 합니다.
> 반면 전압 글리치는 션트 양단(혹은 VCC 라인)에 **크로우바(crowbar) 트랜지스터로 짧은 단락 펄스** 만 가하면 되므로, 와이어태핑 셋업과 자연스럽게 결합됩니다.


### 실험 환경 (물리적 배선)

```
호스트 PC (Jupyter)
    │
    │  USB ×2
    ├─────────────────────────┬──────────────────────────┐
    ▼                                                    ▼
┌──────────────────────────┐                ┌──────────────────────────┐
│   ChipWhisperer-Lite     │                │  ChipWhisperer-Husky      │
│   (active comm + power)  │                │  (passive wire-tap +      │
│                          │                │   active voltage glitch)  │
└──────────────────────────┘                └──────────────────────────┘
    │ 20-pin 커넥터                               │ 전면 20-pin + 측면 SMA
    │                                            │
    │  HS2  ── CLKIN  (타겟 시스템 클럭 공급)       │  D0       ←  CW308 TRIG  (CW308 GPIO4/TRIG)
    │  TX   ── RX                                │  AUX MCX  ←  CW308 CLKIN (타겟 클럭 분기, 주파수 미상)
    │  RX   ── TX                                │  GLITCH OUT (crowbar) → 타겟 VCC 라인 (전압 글리치 주입)
    │  ...                                       │
    ▼                                            ▼
        ┌────────────────────────────────────────────┐
        │   CW308 UFO 보드 + STM32F303 (타겟 MCU)    │
        │                                            │
        │   ※ 본 실습 편의를 위해 타겟 펌웨어 내부에   │
        │      암호화 구간 진입 시 GPIO4 를 TRIG 로    │
        │      토글하도록 구현되어 있습니다.           │
        │      실제 공격에서는 통신 신호(UART idle,    │
        │      특정 패턴 등)를 트리거로 활용해야 합니다.│
        └────────────────────────────────────────────┘
```

**연결 핵심 4선**

| 라인 | 출처/입력 | Husky 측 | 의미 |
|:----:|:----:|:----:|:----|
| 트리거       | 타겟 GPIO4 / TRIG | **전면 20-pin D0**      | 캡처·글리치 시점 정렬용 디지털 신호 |
| 클럭         | 타겟 CLKIN        | **전면 AUX MCX**        | 타겟 동작 클럭 (정확한 주파수 미상 → 카운터로 측정) |
| 전압 글리치 출력 | (Husky → 타겟)  | **GLITCH OUT (crowbar)**| 타겟 VCC 라인에 순간 단락 펄스 주입 |
| (선택) 전력  | 타겟 SHUNTL       | **측면 Measure (Pos)**  | 글리치 효과 사후 분석용 (옵션) |


### 📖 핵심 용어집

| 용어 | 의미 |
|:----|:----|
| **와이어태핑 (Wire-Tapping)** | 통신·연산에 개입하지 않고 신호선만 분기해 측정하는 수동 관측 방식 |
| **multi-device 환경** | 호스트 PC에 두 대 이상의 ChipWhisperer가 동시 연결된 상태 |
| **장비 시리얼 상수** | `HUSKY_SERIAL_NUMBER`와 `LITE_SERIAL_NUMBER`로 두 장비 역할을 고정 |
| **`cw.scope(sn=...)`** | 시리얼 넘버 지정으로 특정 장치에 명시적으로 연결 |
| **`extclk_aux_io`** | PLL 입력 클럭 소스로 전면 AUX MCX 입력을 사용 |
| **`freq_ctr`** | Husky 내장 주파수 카운터의 측정값 (실시간 외부 클럭 측정) |
| **`adc_mul`** | 타겟 클럭 대비 ADC 샘플레이트 배수 (1 → 1 클럭당 1 샘플) |
| **PLL Lock** | Husky 내부 PLL이 외부 클럭에 위상·주파수 정렬 완료 상태 |
| **Golden Model** | 호스트 PC 측에서 직접 계산한 기준 출력값 (펌웨어/통신·연산 검증용) |
| **Voltage Glitch / Crowbar** | VCC 라인을 짧게 단락시키는 외란 펄스 (전압 글리치) |
| **`vglitch_setup(...)`** | 전압 글리치 모듈(고전류 HP / 저전류 LP 트랜지스터) 활성화 함수 |
| **`glitch.ext_offset`** | 트리거 신호 후 **N 클럭** 뒤에 글리치 시작 (단위: clock cycle) |
| **`glitch.offset`**     | 1 클럭 내에서 **언제** 글리치 펄스를 시작할지 (위상; 음수 허용 시 LOW 구간 활용) |
| **`glitch.width`**      | 1 클럭 내에서 글리치 펄스를 **얼마나 오래** 유지할지 (펄스 폭) |
| **`glitch.repeat`**     | 한 번의 글리치 명령에서 연속 펄스를 반복할 횟수 (버스트 길이) |
| **`glitch.num_glitches`** | 한 번의 암호화 시행에서 글리치를 떨어뜨릴 클럭 위치 개수 |
| **`phase_shift_steps`** | offset/width 가 가질 수 있는 **최대 단계 수** (Husky 기준 ≈ 4592) |


---

# 📦 1단계 — 자기완결 설정 및 지정 장치 연결

> **이 단계의 목표**
> 이 노트북에 필요한 패키지와 헬퍼를 직접 정의하고, 배선된 Lite와 Husky만 노트북 전용 시리얼 번호로 연결합니다. 지정 장비 연결에 실패하면 다른 USB 장비로 전환하지 않습니다.

---

### 1.1 패키지·상수·헬퍼 정의

아래 설정 셀은 SimpleSerial 통신, STM32F3 리셋, 오류주입 결과 집계에 필요한 최소 기능만 포함합니다. `HUSKY_SERIAL_NUMBER`와 `LITE_SERIAL_NUMBER`가 각 장비 역할의 단일 기준입니다.

In [1]:
import logging
import random
import subprocess
import time
from pathlib import Path

import chipwhisperer as cw
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

PLATFORM = 'CW308_STM32F3'
CRYPTO_TARGET = 'NONE'
SS_VER = 'SS_VER_2_1'
HUSKY_SERIAL_NUMBER = '502032204c5846303130313137313032'
LITE_SERIAL_NUMBER = '44203120394d36433130322030313035'

FIRMWARE_DIR = Path('simpleserial_main')
FIRMWARE_PATH = FIRMWARE_DIR / f'simpleserial-base-{PLATFORM}.hex'
BUILD_OPTIONS = (
    f'PLATFORM={PLATFORM}',
    f'CRYPTO_TARGET={CRYPTO_TARGET}',
    f'SS_VER={SS_VER}',
)

def my_fsr_cmd(target, cmd, scmd, data, payload_only=False):
    """SimpleSerial 2.1 명령을 보내고 500 ms 안에 받은 응답을 반환한다.

    ``target``은 Lite에 연결된 SimpleSerial2 타겟, ``cmd``는 명령 바이트,
    ``scmd``는 한 문자, ``data``는 페이로드다. ``payload_only``가 참이면
    프레임을 제외한 페이로드를, 거짓이면 응답 전체를 반환한다. 시간 안에
    응답이 없으면 ``None``, 프레임이 짧거나 길이 필드가 틀리면
    ``ValueError``가 발생하고 하위 통신 예외는 그대로 전파된다. 송신 전에
    수신 버퍼를 비우고 타겟 상태를 명령에 맞게 변경한다.
    """
    target.flush()
    target.send_cmd(cmd=cmd, scmd=ord(scmd), data=data)
    response = target.read_cmd(timeout=500)
    if response is None:
        return None
    if len(response) < 3:
        raise ValueError(f'SimpleSerial 응답 프레임이 너무 짧습니다: {response!r}')

    payload_end = 3 + response[2]
    if len(response) < payload_end:
        raise ValueError(f'SimpleSerial 응답 길이가 올바르지 않습니다: {response!r}')
    return response[3:payload_end] if payload_only else response


def reset_target(scope, delay=0.25):
    """Lite의 nRST 핀으로 STM32F3 타겟을 리셋한다.

    ``scope``는 타겟에 연결된 Lite이고 ``delay``는 low와 high-Z 상태를 각각
    유지할 초 단위 시간이다. 반환값은 없다. USB 연결이나 핀 제어가 실패하면
    ChipWhisperer 예외가 그대로 전파된다. nRST 핀과 타겟 실행 상태를 변경하고
    두 번 대기한다.
    """
    scope.io.nrst = 'low'
    time.sleep(delay)
    scope.io.nrst = 'high_z'
    time.sleep(delay)


STATUS = {
    'fail_Encrypt': '[실패] 암호연산 실패',
    'fail_Encrypt_Infinite_loop': '[실패] 암호연산 실패 (무한루프)',
    'fail_husky_capture': '[실패] Husky 캡처 실패',
    'fail_normal': '[실패] 오류주입 실패 (정상동작)',
    'success_FA': '[후보] 골든 모델과 다른 출력',
}
df = pd.DataFrame({'횟수': 0, '의미': list(STATUS.values())}, index=STATUS)
df.index.name = '상태'


def log_init():
    """오류주입 분류 횟수를 0으로 초기화하고 현재 표를 출력한다.

    입력과 반환값은 없다. 전역 ``df``가 없으면 ``NameError``가 발생한다.
    전역 DataFrame을 변경하고 현재 Jupyter 출력을 지운 뒤 새 표를 표시한다.
    """
    df['횟수'] = 0
    clear_output(wait=True)
    display(df)


def log(status, i_ext_offset, i_offset, i_width):
    """한 시행의 분류 횟수와 사용한 글리치 파라미터를 표시한다.

    ``status``는 ``STATUS``의 키이고 나머지 세 값은 ext_offset, offset,
    width다. 반환값은 없다. 알 수 없는 상태면 ``KeyError``가 발생한다.
    전역 ``df``를 변경하고 현재 Jupyter 출력을 새 표와 파라미터로 교체한다.
    """
    df.loc[status, '횟수'] += 1
    clear_output(wait=True)
    display(df)
    print(f'husky_scope.glitch.ext_offset = {i_ext_offset}')
    print(f'husky_scope.glitch.offset     = {i_offset}')
    print(f'husky_scope.glitch.width      = {i_width}')

/usr/local/lib/python3.12/site-packages/chipwhisperer/capture/trace/TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


### 1.2 지정 Lite·Husky 직접 연결

두 장비는 USB 탐색 순서나 제품 이름이 아니라 각 상수의 시리얼 번호로 직접 엽니다. Lite는 타겟 통신·프로그래밍·클럭 공급을 담당하고, Husky는 외부 배선을 통한 수동 측정과 전압 글리치 주입을 담당합니다. 어느 연결도 다른 장비로 fallback하지 않습니다.

In [2]:
# 이 실습에 배선된 두 장비만 직접 연결한다. 어느 한쪽이라도 실패하면
# 다른 USB 장비로 전환하지 않으며, 이미 열린 연결만 닫고 예외를 전파한다.
lite_scope = None
husky_scope = None
try:
    lite_scope = cw.scope(sn=LITE_SERIAL_NUMBER)
    husky_scope = cw.scope(sn=HUSKY_SERIAL_NUMBER)
except Exception:
    if husky_scope is not None:
        husky_scope.dis()
    if lite_scope is not None:
        lite_scope.dis()
    raise

scopes = {
    'ChipWhisperer_Lite': lite_scope,
    'ChipWhisperer_Husky': husky_scope,
}
print(f'[✓] ChipWhisperer_Husky 연결 완료  (SN: {HUSKY_SERIAL_NUMBER})')
print(f'[✓] ChipWhisperer_Lite 연결 완료  (SN: {LITE_SERIAL_NUMBER})')

[✓] ChipWhisperer_Husky 연결 완료  (SN: 502032204c5846303130313137313032)
[✓] ChipWhisperer_Lite 연결 완료  (SN: 44203120394d36433130322030313035)


---

# 🔌 2단계 — Lite를 통한 타겟 보드 통신 채널 확보

> **이 단계의 목표**
> 타겟 보드(STM32F303)와의 **모든 시리얼 통신은 Lite가 전담** 합니다.
> SimpleSerial 프로토콜 객체를 `lite_scope` 위에 바인딩해 `target` 객체를 생성합니다.

---

본 노트북은 **SimpleSerial v2.1** 를 사용합니다.

`cw.target(lite_scope, target_type)` 호출은 다음을 수행합니다:
- Lite의 UART 핀(TX/RX)을 SimpleSerial2 송수신용으로 초기화
- 타겟과의 baud rate 협상 및 동기 바이트 확인
- 향후 모든 `target.send_cmd()` / `read_cmd()` 호출의 경로를 **Lite 경유** 로 고정

> ⚠️ **target 은 lite_scope 에만 묶인다**
> 이후 등장하는 `target.send_cmd(...)`, `my_fsr_cmd(target, ...)` 등 모든 통신은 Lite를 통해 흐릅니다.
> Husky 는 이 통신을 외부에서 **수동 관측 + 전압 글리치 주입** 만 하며, 절대 UART 통신에는 개입하지 않습니다.


In [3]:
# UART 통신은 지정 Lite에만 바인딩한다. Husky는 통신에 개입하지 않는다.
target = cw.target(lite_scope, cw.targets.SimpleSerial2)
print('[✓] ChipWhisperer_Lite에 SimpleSerial2 타겟 연결 완료')

[✓] ChipWhisperer_Lite에 SimpleSerial2 타겟 연결 완료


---

# 🛠 3단계 — 펌웨어 빌드 및 Lite 경유 타겟 프로그래밍

> **이 단계의 목표**
> `simpleserial_main/` 디렉터리의 펌웨어 소스를 STM32F303 용으로 컴파일하고, Lite의 SWD/JTAG 프로그래밍 기능을 이용해 타겟 보드에 플래싱합니다.

---

멀티 디바이스 환경에서는 *어느 장치가 프로그래머 역할인지* 가 중요한 셋업 정보이므로, 이를 셀에서 명시적으로 다루는 편이 이해하기 쉽기 때문입니다.

| 하위 단계 | 동작 | 비고 |
|:----:|:----|:----|
| ① 컴파일       | `make PLATFORM=... CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1` | `subprocess.run` 으로 호스트 셸 위임 |
| ② 프로그래머 선택 | `cw.programmers.STM32FProgrammer` | STM 계열 타겟용 |
| ③ Lite 기본 셋업 | `lite_scope.default_setup()` | Lite 의 클럭/UART/HS2 정상화 |
| ④ 플래싱       | `cw.program_target(lite_scope, prog, hex_path)` | **Lite 가 프로그래머 역할** |
| ⑤ 빌드 산출물 정리 | `make clean` | 작업 디렉터리 청결 유지 |

> 💡 **`lite_scope.default_setup()` 의 부수 효과**
> 이 호출은 Lite 측의 게인·ADC 샘플 수·트리거 모드를 표준값으로 초기화합니다.
> 동시에 Lite 가 HS2 핀으로 타겟에 클럭을 *공급하기 시작* 하는 시점이기도 합니다.
> 이 설정은 **Husky 측 설정과는 완전히 독립적** 이며, 5단계에서 Husky 만 따로 구성하게 됩니다.

> ⚠️ **컴파일 출력 확인 권장**
> `subprocess.run(..., capture_output=True)` 로 컴파일 결과를 캡처하므로 화면에는 진행/완료 메시지만 표시됩니다.
> 컴파일 자체가 실패해도 다음 단계로 넘어갈 수 있으니, **3단계 마지막에 출력되는 `[✓] ... 프로그램 업로드 완료`** 메시지를 반드시 확인하세요.


In [4]:
# 같은 옵션으로 clean한 뒤 빌드해 다른 플랫폼·프로토콜의 낡은 산출물이
# 프로그래밍되는 일을 막는다. 빌드나 플래싱 실패는 즉시 예외로 보고한다.
subprocess.run(['make', *BUILD_OPTIONS, 'clean'], cwd=FIRMWARE_DIR, check=True)
subprocess.run(['make', *BUILD_OPTIONS], cwd=FIRMWARE_DIR, check=True)

lite_scope.default_setup()
try:
    cw.program_target(
        lite_scope,
        cw.programmers.STM32FProgrammer,
        str(FIRMWARE_PATH),
    )
    reset_target(lite_scope, delay=0.05)
    target.flush()
    print(f'[✓] {PLATFORM} 타겟 빌드·프로그램 업로드 완료')
finally:
    # 실행에 필요한 바이너리는 이미 MCU에 기록됐으므로 호스트 산출물은 정리한다.
    subprocess.run(['make', *BUILD_OPTIONS, 'clean'], cwd=FIRMWARE_DIR, check=True)

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
Welcome to another exciting ChipWhisperer target build!!
.
Cleaning project:
rm -f -- simpleserial-base-CW308_CC2538.hex simpleserial-base-CW301_AVR.hex simpleserial-base-CW303.hex simpleserial-base-CW304.hex simpleserial-base-CW308_MEGARF.hex simpleserial-base-CW308_SAM4L.hex simpleserial-base-CW308_STM32F0.hex simpleserial-base-CW308_STM32F1.hex simpleserial-base-CW308_STM32F2.hex simpleserial-base-CW308_STM32F3.hex simpleserial-base-CW308_STM32F4.hex simpleserial-base-CW308_K24F.hex simpleserial-base-CW308_NRF52.hex simpleserial-base-CW308_AURIX.hex simpleserial-base-CW308_SAML11.hex simpleserial-base-CW308_EFM32TG11B.hex simpleserial-base-CWLITEARM.hex simpleserial-base-CWLITEXMEGA.hex simpleserial-base-CWNANO.hex simpleserial-base-CWHUSKY.hex simpleserial-base-CW308_K82F.hex simpleserial-base-CW308_PSOC62.hex simpleserial-base-CW308_IMXRT1062.hex simpleserial-base-CW308_FE310.hex simpleserial-base-CW308_EFR32MG21A.hex simpleseria

---

# ✅ 4단계 — 통신 검증 (Golden Model 비교)

> **이 단계의 목표**
> 글리치 스윕 전에 Lite ↔ 타겟의 요청·응답이 이 실습의 골든 모델과 일치하는지 확인합니다. 이 검사는 사용한 입력의 사전 점검이며 통신 경로 전체의 무결성을 보증하지 않습니다.
> 통신이 불안정하면 이후 수집된 글리치 결과(정상/오류)의 분류 자체가 무의미해지므로, 이 단계의 통과는 필수 사전 조건입니다.

---

타겟 펌웨어는 다음의 단순 연산을 수행합니다:

```c
for (i = 0; i < global_len; i++) {
    output[i] = key[i] ^ plaintext[i];
}
```

호스트에서 동일한 `k ⊕ p` 를 직접 계산한 결과가 **골든 모델** 이며, 정상 동작 시 타겟의 반환값과 바이트 단위로 일치해야 합니다.
이후 6단계에서는 이 골든 모델과 다른 출력이 나오면 **글리치가 연산에 영향을 미친 시도(success_FA)** 로 판정합니다.

| 명령 코드 | scmd | 의미 |
|:----:|:----:|:----|
| `0x81` | `'k'` | 키 주입 |
| `0x81` | `'p'` | 평문 주입 |
| `0x81` | `'l'` | 출력 길이 통보 (`MAX_DATA_LEN`) |
| `0x82` | `'c'` | 연산 트리거 (펌웨어가 GPIO4 토글) |
| `0x83` | `'r'` | 결과 회수 |

> 💡 **이 검증이 실패하는 주요 원인**
> - SimpleSerial 버전 불일치 (펌웨어가 v1 으로 빌드되었는데 호스트는 v2 사용)
> - Lite 의 UART 핀 매핑 오류 (`default_setup()` 호출 누락)
> - 타겟의 `global_len` 미설정 (`scmd='l'` 명령 누락)
> - 펌웨어 플래싱 실패 (3단계의 `[✓]` 표시 확인)

> 🔬 **`MAX_DATA_LEN = 100` 의 의미**
> 본 노트북은 한 번의 연산에서 100 바이트의 `k ⊕ p` 를 처리하도록 설정합니다.
> 이는 글리치가 *for-loop 본체* 중 어디에 떨어졌는지에 따라 결과 바이트 패턴이 어떻게 달라지는지 더 풍부한 사례를 확보하기 위함입니다.


In [5]:
MAX_DATA_LEN = 100

# 재현성을 위해 시드 고정
random.seed(1)

# 무작위 키(k), 평문(p) 생성 후 호스트에서 사전 계산한 골든 결과(k XOR p)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

# 0x81 = 데이터 전송 명령 ('k'=key, 'p'=plaintext, 'l'=length)
# 0x82 = 연산 트리거 명령
# 0x83 = 결과 회수 명령
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
my_fsr_cmd(target, 0x82, 'c', [])
Return_k_XOR_p = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

print('=== 결과 비교 ===')
target_hex = Return_k_XOR_p.hex(' ') if Return_k_XOR_p is not None else '<응답 없음>'
print(f'타겟 결과 : {target_hex}')
print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
print()
if Golden_k_XOR_p == Return_k_XOR_p:
    print('[✓] 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
else:
    raise RuntimeError('타겟 출력이 골든 모델과 다릅니다. 통신·펌웨어를 확인하세요.')

=== 결과 비교 ===
타겟 결과 : 1e 7c ac be ed c2 db ca 8c 37 76 71 4e e5 5f 54 1c 56 61 f3 8d 5d 86 92 2b ca 26 fc 4b ec 62 b2 1f a1 10 6f a3 8a be ee 55 e4 34 61 92 73 c9 75 25 b9 f7 87 73 04 70 82 a5 27 6c 8c 2f cf ed 9a 64 97 a1 c3 1c 8b 53 45 1b ed 31 65 0e d6 05 d9 e0 df a7 be 5c 1d 8b 09 b2 c0 1e 68 bf 84 d7 a7 ec 47 d5 18
골든 모델 : 1e 7c ac be ed c2 db ca 8c 37 76 71 4e e5 5f 54 1c 56 61 f3 8d 5d 86 92 2b ca 26 fc 4b ec 62 b2 1f a1 10 6f a3 8a be ee 55 e4 34 61 92 73 c9 75 25 b9 f7 87 73 04 70 82 a5 27 6c 8c 2f cf ed 9a 64 97 a1 c3 1c 8b 53 45 1b ed 31 65 0e d6 05 d9 e0 df a7 be 5c 1d 8b 09 b2 c0 1e 68 bf 84 d7 a7 ec 47 d5 18

[✓] 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)


---

# 📡 5단계 — Husky 측 와이어태핑 + 전압 글리치 환경 구성

> **이 단계의 목표**
> 통신·프로그래밍·클럭 공급은 Lite 가 이미 안정적으로 수행 중인 상태에서,
> **Husky 가 통신에는 일체 개입하지 않으면서** (a) 트리거/클럭/(선택)전력 라인을 정확히 동기 측정하고, (b) 동시에 전압 글리치 펄스를 주입할 수 있도록 구성합니다.
> 본 노트북의 핵심 셋업 단계이며, 이하 모든 설정은 오직 `husky_scope` 에만 적용됩니다.

---

### 5.1 전압 글리치 모듈 활성화 + 외부 클럭 입력 모드 전환

본 셀은 두 가지를 동시에 수행합니다.

**① 전압 글리치 모듈(`vglitch_setup`) 활성화**

`vglitch_setup('both')` 는 Husky 내부의 **HP(High-Power) + LP(Low-Power) 크로우바 트랜지스터** 를 모두 글리치 출력에 활성화합니다.
크로우바는 글리치 펄스가 들어오는 짧은 시간 동안 타겟 VCC 라인을 GND 로 끌어내려, 회로에 순간적인 전압 강하를 일으킵니다.

| 함수 호출 | 활성 트랜지스터 | 특징 |
|:----|:----|:----|
| `vglitch_setup('hp')`   | HP 만 | 강한 글리치 (Husky 환경에서 단독으로 잘 동작) |
| `vglitch_setup('lp')`   | LP 만 | 약한 글리치 |
| `vglitch_setup('both')` | HP + LP 동시 | 가장 강한 외란 (본 노트북에서 사용) |

`default_setup=False` 를 전달하는 두 번째 호출은, 첫 번째 호출이 내부적으로 적용한 일부 default 값(트리거/ADC 설정)을 *재초기화 없이* 트랜지스터만 갈아 끼우기 위함입니다.

**② 외부 클럭 입력 준비 (와이어태핑 클럭 동기화 1단계)**

부채널·FIA 측정의 신호 품질은 **ADC/글리치 클럭이 타겟 클럭과 얼마나 정밀히 동기화되었느냐** 에 의해 결정됩니다 (지터 최소화).
단일 Husky 구성은 Husky가 직접 타겟에 클럭을 공급했으므로 동기화가 자동이었지만, 본 노트북에서 Husky는 **외부에서 클럭 신호를 받아오는 입장** 입니다.

따라서 Husky 는 아래의 단계로 외부 클럭을 **탐색·정렬** 해야 합니다.

```
[1] PLL 입력 소스를 외부 AUX 로 전환                  ← 본 단계
[2] AUX MCX 핀을 high-Z 입력 모드로 설정              ← 본 단계
[3] 내장 주파수 카운터로 외부 클럭 주파수 측정         ← 다음 단계
[4] 측정된 주파수로 PLL의 목표 주파수 설정            ← 다음 단계
[5] ADC 클럭을 타겟 클럭과 동일하게 (adc_mul=1)       ← 다음 단계
[6] ADC 리셋 후 lock 상태 확인                        ← 다음 단계
```

> ⚠️ **`clkgen_freq = 0` 의 의미**
> PLL 의 목표 주파수를 `0` 으로 강제 해제해 *현재 잠금 상태를 떨어뜨리는* 안전한 초기화입니다.
> 이어지는 `clkgen_src = 'extclk_aux_io'` 가 실제 입력 소스를 바꾸기 직전, 잘못된 주파수에 PLL이 계속 잠겨 있지 않도록 비워두는 과정입니다.


In [6]:
# 이전 설정의 간섭을 방지하기 위해 Husky 공장 초기화
husky_scope.default_setup()
time.sleep(0.5)

# 양쪽 크로우바 출력으로 전압 글리치 모드를 한 번만 초기화한다.
husky_scope.vglitch_setup('both')
time.sleep(0.5)

print("Husky 스코프 하드웨어 동기화 및 튜닝 중...")

# ---------------------------------------------------------
# [1] 클럭 신호 탐색 및 동기화 (Aux in/out)
# ---------------------------------------------------------
husky_scope.clock.clkgen_freq = 0
husky_scope.clock.reset_adc()
# AUX MCX 를 입력(high-Z)으로 설정 → Husky 가 클럭을 driving 하지 않고 수신만 함
husky_scope.io.aux_io_mcx = 'high_z'
# PLL 입력 소스를 외부 클럭(extclk)으로 지정
husky_scope.clock.clkgen_src = 'extclk_aux_io'
husky_scope.clock.reset_adc()

if (husky_scope.io.aux_io_mcx == 'high_z') and (husky_scope.clock.clkgen_src == 'extclk_aux_io'):
    print(f"[✓] io.aux_io_mcx     = {husky_scope.io.aux_io_mcx}")
    print(f"[✓] clock.clkgen_src  = {husky_scope.clock.clkgen_src}")
else:
    raise RuntimeError('외부 클럭 설정 실패')

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_shift_steps           changed from 0                         to 4592                     
scope.trace.capture

### 5.2 외부 클럭 주파수 탐색 + ADC 1× 동기화 (1 sample = 1 clock)

핵심 한 줄은 다음입니다:

```python
husky_scope.clock.clkgen_freq = data.mode().iloc[0]
```

`freq_ctr` 는 Husky 내장 주파수 카운터가 실시간으로 측정한 외부 클럭 값입니다.
**0.2초 간격으로 20회 측정한 뒤 최빈값(mode) 을 PLL 목표 주파수로 그대로 대입** 함으로써, 카운터의 일시적 흔들림을 통계적으로 제거하고 가장 신뢰할 만한 한 점을 잡습니다.

```
        외부 클럭 측정 → 통계 처리 → PLL 잠금
        ────────────────────────────────────
        freqs = [f₀, f₁, ..., f₁₉]   (20회 측정)
                       │
                       ▼
              data.mode().iloc[0]    (최빈값 = 가장 안정한 한 점)
                       │
                       ▼
        husky_scope.clock.clkgen_freq = <최빈값>
```

| 설정 | 값 | 의미 |
|:----:|:----:|:----|
| `pll._allow_rdiv`   | `True` | PLL의 reference divider 분수 합성 허용 → 더 정밀한 주파수 추적 |
| `freq_ctr_src`      | `'extclk'` | 카운터의 측정 대상으로 외부 클럭 지정 |
| `clkgen_freq`       | `freq_ctr` 의 최빈값 (≈ 7.4 MHz) | 측정된 주파수에 PLL 목표 잠금 |
| `adc_mul`           | `1` | **1 클럭 → 1 ADC 샘플 (FIA용 시점 식별에 적합)** |
| `adc.decimate`      | `1` | 다운샘플링 없음 |

> 🔬 **`adc_mul = 1` 이 FIA 에서 중요한 이유**
> 부채널 측정(SCA)에서는 `adc_mul = 4` 등 오버샘플링으로 미세 누설 패턴을 잡는 것이 유리합니다.
> 그러나 **FIA 에서는 "몇 번째 클럭에 글리치를 떨어뜨릴지"** 가 가장 중요한 정보이므로, `1 ADC sample = 1 target clock` 정렬이 필수입니다.
>
> ```
> ADC 샘플레이트 = 타겟 클럭 × adc_mul × (1/decimate)
>   · adc_mul = 4   → 1 클럭당 4 샘플 (SCA에서 미세 누설 분석용)
>   · adc_mul = 1   → 1 클럭당 1 샘플 (FIA에서 시점 식별용) ← 본 노트북
> ```
>
> 이렇게 정렬하면 `scope.adc.trig_count` 가 곧 *연산이 차지한 클럭 사이클 수* 가 되며, `glitch.ext_offset = N` 으로 설정하면 N 번째 클럭에 정확히 글리치가 떨어지는 직관적 매핑이 성립합니다.

> ⚠️ **PLL이 정확한 주파수에 잠기지 않을 수 있음**
> `husky_scope.clock.pll._allow_rdiv = True` 을 주석처리하면, 위 셀의 출력에서
> `Could not calculate pll settings for the requested frequency (7384506); generating a 7400000 clock instead.`
> 와 같은 메시지가 보일 수 있습니다.
> Husky의 PLL이 임의의 분수 주파수를 정확히 합성하지 못해 **가장 가까운 합성 가능 주파수** 로 대체한다는 의미입니다.
> 본 코드는 `_allow_rdiv = True` 로 정밀 합성을 활성화해 이 문제를 회피합니다.


In [7]:
# ---------------------------------------------------------
# [2] 외부 클럭 주파수 탐색 및 동기화
# ---------------------------------------------------------
# 더 정밀한 클럭 주파수 설정 가능 (동기화 및 PLL 잠금이 실패할 경우에 주석처리하면 동작할 수 있습니다)
husky_scope.clock.pll._allow_rdiv = True
# 주파수 카운터의 측정 대상을 외부 클럭으로 지정
husky_scope.clock.freq_ctr_src = 'extclk'
# 카운터 초기 안정화를 위한 짧은 대기
time.sleep(0.5)

# 주파수 데이터 수집 (0.2초 간격, 20회)
freqs = []
for _ in range(20):
    freqs.append(husky_scope.clock.freq_ctr)
    time.sleep(0.2)
data = pd.Series(freqs)
print(f"최빈값: {data.mode().iloc[0]} (등장 {(data == data.mode().iloc[0]).sum()}/{len(data)}회)")
print(f"범위: {data.min()} ~ {data.max()} (Δ={data.max()-data.min()})")
print("\n[전체 통계 요약]")
print(data.describe()) # 개수, 평균, 표준편차, 최소, 최대, 사분위수 출력

# 내부/외부 클럭 주파수 동기화
husky_scope.clock.clkgen_freq = data.mode().iloc[0]
# ADC 샘플레이트 클럭 동기화: 1 샘플 = 1 클럭
husky_scope.clock.adc_mul    = 1            # 오버샘플링 없음
husky_scope.adc.decimate     = 1            # 다운샘플링 없음
# ADC 리셋
husky_scope.clock.reset_adc()

if husky_scope.clock.adc_locked:
    print("[✓] ADC 클럭 동기화 완료")
    print(f"   - ADC 샘플레이트 (adc_freq): {husky_scope.clock.adc_freq:,.0f} Hz")
else:
    raise RuntimeError('ADC 클럭 동기화 실패: 외부 클럭과 ADC 상태를 확인하세요.')

if husky_scope.clock.clkgen_locked:
    print("[✓] Husky PLL 잠금 성공")
    print(f"   - 타겟 클럭 (clkgen_freq) : {husky_scope.clock.clkgen_freq:,.0f} Hz")
else:
    raise RuntimeError('Husky PLL 잠금 실패: 외부 클럭의 진폭·듀티·안정성을 확인하세요.')

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:585) scope.clock.pll._allow_rdiv is True; this can cause an inconsistant phase relationship between the target and sampling clocks. Do you really want this?


최빈값: 7384494 (등장 11/20회)
범위: 7384494 ~ 7384506 (Δ=12)

[전체 통계 요약]
count    2.000000e+01
mean     7.384499e+06
std      6.125013e+00
min      7.384494e+06
25%      7.384494e+06
50%      7.384494e+06
75%      7.384506e+06
max      7.384506e+06
dtype: float64


(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:585) scope.clock.pll._allow_rdiv is True; this can cause an inconsistant phase relationship between the target and sampling clocks. Do you really want this?


[✓] ADC 클럭 동기화 완료
   - ADC 샘플레이트 (adc_freq): 7,384,494 Hz
[✓] Husky PLL 잠금 성공
   - 타겟 클럭 (clkgen_freq) : 7,384,494 Hz


### 5.3 트리거 입력 핀 설정 (전면 USERIO D0)

타겟 펌웨어는 암호화 진입 시 GPIO4 라인을 LOW → HIGH 로 토글합니다.
이 신호를 Husky 전면 20-pin의 **D0** 으로 받아들여, **상승 엣지(rising edge)** 에 캡처와 글리치를 동시에 트리거합니다.

| 설정 | 값 | 의미 |
|:----:|:----:|:----|
| `trigger.triggers` | `'userio_d0'` | 전면 USERIO D0 핀을 트리거 입력으로 사용 |
| `trigger.module`   | `'basic'`     | 단순 엣지/레벨 검출 모듈 |
| `adc.basic_mode`   | `'rising_edge'` | 상승 엣지에서 캡처 시작 |

> 💡 **이 트리거 신호가 글리치 시점의 기준**
> 이후 6단계에서 설정하는 `glitch.ext_offset = N` 은 **이 트리거의 상승 엣지로부터 N 클럭 뒤** 에 글리치 펄스를 시작하라는 의미입니다.
> 따라서 트리거 신호가 부정확하거나 지터가 크면 글리치 시점도 흔들리며, 같은 파라미터로도 결과가 달라집니다.
> 타겟 펌웨어의 GPIO4 토글이 **연산 시작과 가장 가까운 시점** 에 놓여 있는지 항상 확인하세요.


In [8]:
# ---------------------------------------------------------
# [3] 트리거 핀 설정 (전면 USERIO - D0 핀)
# ---------------------------------------------------------
# 트리거 입력 소스 = 전면 USERIO D0
husky_scope.trigger.triggers = 'userio_d0'
# 단순 엣지/레벨 검출용 'basic' 트리거 모듈 사용
husky_scope.trigger.module = 'basic'
# 캡처 시작 조건 = 상승 엣지 (타겟이 트리거를 LOW → HIGH 로 토글)
husky_scope.adc.basic_mode = 'rising_edge'

print("[✓] Husky 스코프 파라미터 설정 완료")
print(f"trigger.triggers = {husky_scope.trigger.triggers}")
print(f"trigger.module   = {husky_scope.trigger.module}")
print(f"adc.basic_mode   = {husky_scope.adc.basic_mode}")

[✓] Husky 스코프 파라미터 설정 완료
trigger.triggers = userio_d0
trigger.module   = basic
adc.basic_mode   = rising_edge


### 5.4 ADC 캡처 파라미터 (게인 / 샘플 수 / 오프셋)

글리치 전후의 전압 변화를 담는 Trace를 후속 분석을 위해 캡처합니다.
션트 양단(혹은 VCC 라인)의 차동 입력은 측면 **Measure (Pos/Neg)** 핀을 통해 곧바로 Husky 내부 LNA → ADC 경로로 흘러갑니다.

| 파라미터 | 값 | 설명 |
|:----:|:----:|:----|
| `gain.db`        | `25` (dB) | LNA 게인. 너무 높으면 클리핑, 너무 낮으면 SNR 저하 |
| `adc.samples`    | `2000`    | 한 번의 캡처에서 수집할 샘플 개수 |
| `adc.offset`     | `0`       | 트리거 후 캡처 시작점 (0 = 트리거 즉시 시작) |
| `adc.presamples` | `0`       | 트리거 직전에 추가로 수집할 샘플 수 |

> 🔬 **`adc.samples = 2000` 의 의미**
> 5.2 에서 `adc_mul = 1` 로 정렬했으므로, **2000 샘플 = 2000 클럭** 입니다.
> `2000`은 연산 구간과 글리치 이후 구간을 함께 관찰하기 위한 초기값입니다. 펌웨어·컴파일 옵션·클럭 설정에 따라 실행 길이가 달라질 수 있으므로, 실제 Trace에서 연산 종료와 잘림 여부를 확인한 뒤 조정해야 합니다.
> SCA 와이어태핑 노트북(10000 샘플 × 4-배 오버샘플링)과 다른 값을 쓰는 이유가 여기에 있습니다.

> 💡 **`gain.db = 25` 의 양자화**
> 입력값은 사용자가 25 를 요청하지만, Husky의 게인 단계가 이산값(quantized)이므로 실제로는 약 `25.09` 처럼 가까운 값에 맞춰집니다.
> 이는 정상이며, 출력 메시지로 실제 적용된 값을 확인할 수 있습니다.


In [9]:
# LNA 게인 (dB). 보통 20~30 dB 사이. 너무 높으면 클리핑, 너무 낮으면 SNR 저하.
husky_scope.gain.db = 25

# 한 번의 캡처에서 수집할 샘플 개수
husky_scope.adc.samples = 2000

# 트리거 이후 캡처 시작점 (0 = 트리거 즉시 캡처 시작)
husky_scope.adc.offset = 0

# 트리거 이전 샘플 (사전 캡처). 필요 시 양수로 설정 가능.
husky_scope.adc.presamples = 0

print(f"gain.db          = {husky_scope.gain.db}")
print(f"adc.samples      = {husky_scope.adc.samples}")
print(f"adc.offset       = {husky_scope.adc.offset}")
print(f"adc.presamples   = {husky_scope.adc.presamples}")

gain.db          = 25.091743119266056
adc.samples      = 2000
adc.offset       = 0
adc.presamples   = 0


### 5.5 글리치 출력 모드 및 로그 억제

마지막으로 글리치 파형 합성 방식을 지정합니다.

**① `glitch.output = 'glitch_only'` — 이 실습의 전압 글리치 출력 모드**

Husky 의 글리치 모듈은 다음 5가지 합성 모드를 가집니다:

| 모드 | 동작 | 주 용도 |
|:----|:----|:----|
| `clock_only`  | 입력 클럭만 그대로 출력             | 디버그용 |
| `glitch_only` | **글리치 펄스만 출력 (클럭 X)**     | **전압 글리치 (본 노트북)** |
| `clock_or`    | 클럭 OR 글리치                      | 클럭 글리치 |
| `clock_xor`   | 클럭 XOR 글리치                     | 클럭 글리치 |
| `enable_only` | 지정 클럭 사이클 동안만 클럭 출력   | 카운터 기반 정밀 글리치 |

이 실습에서는 크로우바 트랜지스터의 ON/OFF 펄스만 출력해야 하므로 `glitch_only` 를 사용합니다.

**② `glitch.arm_timing = 'after_scope'` — 글리치 활성 타이밍**

`'after_scope'` 는 `scope.arm()` 호출 직후에 글리치 모듈을 활성화한다는 의미입니다.
즉, 캡처 무장 → 글리치 무장 → 트리거 도착 → 두 동작 동시 실행의 순서가 보장됩니다.
(디버그 시 글리치를 비활성화하려면 `'no_glitch'` 로 두면 됩니다.)

**③ 정상 경고 로그 억제**

파라미터 스윕 도중에는 다음 경고가 발생할 수 있습니다. 탐색 과정에서 예상 가능한 실패이지만, 빈도가 높으면 배선·범위·타겟 복구 절차의 문제일 수 있으므로 별도로 기록해 점검해야 합니다.

- `(ChipWhisperer Target  WARNING) Read timed out` — 글리치가 너무 강해 타겟이 응답을 못 함 (코드 0 freezing 으로 분류)
- `(ChipWhisperer Glitch  WARNING) Partial reconfiguration for width = 0 may not work` — `width = 0` 케이스에서 발생 (스윕 루프에서 미리 `continue` 로 건너뜀)

이 노트북은 화면 갱신량을 줄이기 위해 두 로거의 레벨을 `ERROR`로 올립니다. 이 설정은 경고 원인을 해결하지 않으며, 진단할 때는 로그 레벨을 되돌려 원문을 확인해야 합니다.


In [10]:
# ── 글리치 발생 시점 + 출력 모드 ────────────────────────
husky_scope.glitch.arm_timing = 'after_scope'   # scope arm 후 글리치 활성
husky_scope.glitch.output     = 'glitch_only'   # · glitch_only : 전압 글리치용 

# ── 경고 로그 무시 (탐색 중 빈번하게 발생하는 정상 경고) ──
# (ChipWhisperer Target  WARNING) Read timed out
# (ChipWhisperer Glitch  WARNING) Partial reconfiguration for width = 0 may not work
logging.getLogger('ChipWhisperer Target').setLevel(logging.ERROR)
logging.getLogger('ChipWhisperer Glitch').setLevel(logging.ERROR)

print('[✓] 글리치 출력 모드 설정 완료')
print(f'  arm_timing : {husky_scope.glitch.arm_timing}')
print(f'  output     : {husky_scope.glitch.output}')

[✓] 글리치 출력 모드 설정 완료
  arm_timing : after_scope
  output     : glitch_only


---

# 🌊 6단계 — 전압 글리치 파라미터 스윕 + 와이어태핑 Trace 수집

> **이 단계의 목표**
> `(ext_offset, offset, width)` 의 3차원 글리치 파라미터 공간을 격자(grid) 탐색하며,
> 각 조합마다 **여러 회 반복** 시행해 (a) 골든 모델과 다른 출력값, (b) 그 시행의 와이어태핑 Trace, (c) 사용한 파라미터를 함께 수집합니다. 출력 불일치는 오류주입 후보이며, 글리치가 원인인지와 공격에 유용한 결함인지는 후속 분석으로 확인해야 합니다.
> 이 데이터가 이후 단계의 **DFA(Differential Fault Analysis)** 및 **글리치 누설 분석** 의 원자료가 됩니다.

---

### 6.1 인터랙티브 출력 위젯 구성

`ipywidgets.Output()` 두 개를 가로로 나란히 배치해 **좌측은 통계 로그, 우측은 마지막 캡처 파형** 을 실시간으로 갱신합니다.

```
┌─────────────────────────────┬─────────────────────────────┐
│ out1 (좌)                   │ out2 (우)                   │
│   - log_init() / log(...)   │   - DEBUG 모드 시 마지막    │
│   - 각 결과 유형별 누적 카운트│     wave_husky 를 matplotlib │
│     (fail_*, success_FA 등) │     로 즉시 그려서 확인     │
└─────────────────────────────┴─────────────────────────────┘
```

이 구성 덕분에 수 시간이 걸릴 수 있는 파라미터 스윕 도중에도 **모니터를 닫지 않고 진행 상황을 한눈에** 확인할 수 있습니다.
`DEBUG = True`로 두면 매 시행마다 우측의 Trace 그림을 갱신하고, `False`로 두면 그림 갱신 비용 없이 스윕합니다.

### 6.2 `Encrypt()` 함수의 역할

```python
Encrypt(data_k, data_p) → bytes | None  # 반환: 타겟이 계산한 k ⊕ p (실패 시 None)
```

내부는 5단계로 구성됩니다.

| 단계 | 명령 | 의미 |
|:----:|:----:|:----|
| ① | `0x81 'k'` | 키 주입 |
| ② | `0x81 'p'` | 평문 주입 |
| ③ | `0x81 'l'` | 출력 길이(`MAX_DATA_LEN=100`) 통보 |
| ④ | `0x82 'c'` | 연산 트리거 (펌웨어가 GPIO4 토글 → Husky 캡처/글리치 시작) |
| ⑤ | `0x83 'r'` | 결과 회수 |

### 6.3 글리치 파라미터의 의미와 본 셀의 스윕 범위

```
husky_scope.glitch.repeat       = 3   ← 한 글리치 명령에서 펄스를 3번 연속 발사 (버스트)
husky_scope.glitch.num_glitches = 1   ← 한 암호화 시행에서 글리치가 떨어지는 클럭 위치는 1개
husky_scope.glitch.ext_offset       ← 트리거 후 몇 번째 클럭에 글리치를 떨어뜨릴지
husky_scope.glitch.offset           ← 1 클럭 내에서 언제 펄스를 시작할지 (위상; 음수도 허용)
husky_scope.glitch.width            ← 1 클럭 내 펄스의 폭 (얼마나 오래 VCC 를 끌어내릴지)
```

본 셀은 다음 좁은 영역을 정밀 탐색합니다 (사전 실험으로 후보 영역을 좁힌 후의 미세 조정 단계로 가정):

| 변수 | 범위 | step | 의미 |
|:----:|:----:|:----:|:----|
| `i_ext_offset` | `range(152, 155)`        | 1   | 트리거 후 152~154 클럭 |
| `i_offset`     | `range(-1200, -1300, -200)` | -200 | 1-클럭 내 LOW 구간 (음수) 의 위상 |
| `i_width`      | `range(2065, 2066, 1)`   | 1   | 폭 2065 단계 (≈ 클럭 절반에 가까운 펄스) |

> 💡 **`offset` 이 음수인 이유**
> Husky 의 글리치 모듈은 1 클럭 내 위상을 `0 ~ phase_shift_steps` 범위에서 표현하는데, 본 셀처럼 음수를 사용하면 **클럭 LOW 구간** 의 동일 위상으로 회귀되어 해석됩니다.
> 동일한 폭이라도 클럭 HIGH 에 떨어지느냐 LOW 에 떨어지느냐에 따라 회로의 즉시 응답이 다르므로, 후보군에 음수 위상이 포함되는 것은 흔한 일입니다.

> ⚠️ **유효성 검사 두 줄은 절대 빼지 말 것**
> ```python
> if i_width == 0: continue
> if (i_offset + i_width) > husky_scope.glitch.phase_shift_steps: continue
> ```
> 첫 줄은 `width=0` 시 발생하는 partial reconfiguration 경고를 피합니다.
> 둘째 줄은 `offset + width` 가 클럭 1주기를 넘어가면 글리치가 다음 클럭으로 흘러들어가 의도와 다른 결과를 내기 때문에 의미 없는 조합을 건너뜁니다.

### 6.4 단일 시행의 6단계 흐름

각 `(ext_offset, offset, width)` 조합에서 **5회 반복** 시행하며, 매 시행마다 아래의 흐름을 따릅니다.

```
[1] reset_target(lite_scope)               타겟 초기화 (이전 시행의 부작용 제거)
[2] husky_scope.arm()                      Husky 캡처 + 글리치 동시 무장
[3] Encrypt(data_k, data_p) → ct           Lite 가 통신 → 타겟이 트리거 발생
                                           동시에 Husky 가 글리치를 발사하고 파형 캡처
[4] ret_husky = husky_scope.capture()      캡처 완료 대기
[5] 결과 ct 의 4가지 케이스 분류
       ┌─ ct is None              → "fail_Encrypt"          (통신/응답 자체 실패)
       ├─ ct == b'3\x00'          → "fail_Encrypt_Infinite_loop" (펌웨어가 무한 루프 진입)
       ├─ ct == Golden_k_XOR_p    → "fail_normal"            (글리치가 무효; 정상 출력)
       └─ ct != Golden_k_XOR_p    → "success_FA"             출력 불일치 후보
[6] success_FA 인 경우에만 t_husky/i_k/i_p/o_c/glitch_parameter 에 저장
```

> 🔬 **이 분류의 핵심 통찰**
> 이 셀의 `success_FA`는 골든 모델과 다른 응답을 선별하는 이름이지, 공격 성공 판정이 아닙니다. 통신 오류와 비결정적 오동작도 같은 범주에 들어갈 수 있습니다.
> 반복성, 오류 위치·형태, 정상 대조 시행을 함께 분석해 글리치 유발 결함인지 확인한 뒤에만 DFA나 loop skip 후보로 해석할 수 있습니다.

> 💡 **강의용 반복 횟수가 5인 이유**
> 같은 파라미터의 글리치 효과는 확률적이지만 전수조사급 탐색은 긴 시간이 필요합니다. 이 셀은 하드웨어 연동과 A–Z 실행을 강의 시간 안에 확인하도록 3 × 1 × 1 × 5 = **15회**만 시행합니다. 오류가 반드시 발생한다는 보장은 없으며, 실제 파라미터 탐색과 재현성 평가는 별도의 장시간 실험으로 수행해야 합니다.

### 6.5 본 셀이 수집하는 데이터 자료구조

| 변수 | 형식 | 의미 |
|:----:|:----:|:----|
| `t_husky`          | `list[np.ndarray]` (각 길이 2000) | `success_FA`로 분류된 시행의 Husky 와이어태핑 Trace |
| `i_k`              | `list[bytearray]` | 사용된 키 (모든 시행 동일하지만 후속 호환성을 위해 함께 저장) |
| `i_p`              | `list[bytearray]` | 사용된 평문 |
| `o_c`              | `list[bytes]`     | 타겟이 실제로 반환한 (변조된) 결과 |
| `glitch_parameter` | `list[list[int]]` | `[ext_offset, offset, width]` 의 3-튜플 |

이 5개의 리스트는 **같은 인덱스가 동일한 시행을 가리키도록** 동기 추가됩니다.
즉, `t_husky[i]`의 Trace는 `glitch_parameter[i]`의 파라미터를 사용한 시행에서 수집되었고, 같은 시행의 응답은 `o_c[i]`입니다.


In [11]:
import ipywidgets as widgets

def Encrypt(data_k, data_p):    
    """전역 ``target``에 키와 평문을 보내 XOR 연산 결과 payload를 읽는다.

    두 입력은 각각 ``MAX_DATA_LEN`` 바이트의 SimpleSerial payload다. 키·평문·
    길이를 설정하고 연산을 실행하므로 타겟 상태를 변경한다. 응답 payload를
    반환하며 응답이 없으면 ``None``을 반환한다. 통신 예외는 호출자에게 전달된다.
    """
    my_fsr_cmd(target, 0x81, 'k', data_k)
    my_fsr_cmd(target, 0x81, 'p', data_p)
    my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
    my_fsr_cmd(target, 0x82, 'c', [])     
    ret = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)
    if ret == None:
        return None
    return ret


out1 = widgets.Output()
out2 = widgets.Output()
display(widgets.HBox([out1, out2]))

DEBUG = True        # True False

# 수집 데이터 초기화
t_husky = []
i_k = []
i_p = []
o_c = []
glitch_parameter = []
with out1:
    log_init()

# 고정 시드
random.seed(1)
MAX_DATA_LEN = 100  # 한 번에 전송 가능한 최대 데이터 크기 (바이트)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

# husky_scope.glitch.phase_shift_steps == 4592
# husky_scope.glitch.phase_shift_steps // 2 == 2296

husky_scope.glitch.repeat = 3                   # 하나의 오류주입에 버스트 횟수
husky_scope.glitch.num_glitches = 1             # 하나의 암호화 시행에 오류가 주입되는 클럭 개수

for i_ext_offset in range(152, 155):

    #for i_offset in range(0, husky_scope.glitch.phase_shift_steps, 500):
    for i_offset in range(-1200, -1300, -200):       #range(-1200, -1050, 200)
        
        #for i_width in range(0, husky_scope.glitch.phase_shift_steps // 2, 500):
        for i_width in range(2065, 2066, 1):        #range(2064, 2066, 1)
            # 유효성 검사 — 의미 없는 파라미터 건너뛰기
            if i_width == 0:
                continue
            if (i_offset + i_width) > husky_scope.glitch.phase_shift_steps:
                continue

            # 글리치 파라미터 설정
            husky_scope.glitch.ext_offset = i_ext_offset    # 트리거 후 N 클럭
            husky_scope.glitch.offset     = i_offset        # 1 클럭 내 시작 위상
            husky_scope.glitch.width      = i_width         # 글리치 펄스 폭


            # 3 × 1 × 1 × 5 = 15회로 강의 시간 안의 A–Z 검증에 맞춘다.
            for _ in range(5):
                reset_target(lite_scope)
                husky_scope.arm()
                ct = Encrypt(data_k, data_p)
                ret_husky = husky_scope.capture()

                if ret_husky:
                    with out1:
                        log("fail_husky_capture", husky_scope.glitch.ext_offset, husky_scope.glitch.offset, husky_scope.glitch.width)
                    continue                      
                if DEBUG:
                    with out2:
                        wave_husky = husky_scope.get_last_trace()
                        out2.clear_output(wait=True)
                        plt.plot(wave_husky)     
                        plt.show()                 
                        plt.close('all')             

                if ct == None:
                    with out1:
                        log("fail_Encrypt", husky_scope.glitch.ext_offset, husky_scope.glitch.offset, husky_scope.glitch.width)
                    continue
                if ct == b'3\x00':
                    with out1:
                        log("fail_Encrypt_Infinite_loop", husky_scope.glitch.ext_offset, husky_scope.glitch.offset, husky_scope.glitch.width)
                    continue
                if ct == Golden_k_XOR_p: 
                    with out1:                
                        log("fail_normal", husky_scope.glitch.ext_offset, husky_scope.glitch.offset, husky_scope.glitch.width)                    
                    continue
                           
                    
                wave_husky = husky_scope.get_last_trace()

                t_husky.append(wave_husky)
                i_k.append(data_k)
                i_p.append(data_p)
                o_c.append(ct) 
                glitch_parameter.append(
                        [husky_scope.glitch.ext_offset, husky_scope.glitch.offset, husky_scope.glitch.width]) 
                with out1:
                    log("success_FA", husky_scope.glitch.ext_offset, husky_scope.glitch.offset, husky_scope.glitch.width)

                with out2:
                    print(f"타겟 결과 == 골든 모델 ? {ct == Golden_k_XOR_p}")
                    print(f'타겟 결과 : {ct.hex(" ")}')
                    print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
                    out2.clear_output(wait=True)
                    plt.plot(wave_husky)     
                    plt.show()                 
                    plt.close('all')     

---

# 🔚 7단계 — 다중 장치 자원 해제

> **이 단계의 목표**
> 노트북 종료 전에 **두 스코프와 타겟 객체를 모두 명시적으로 해제** 해 다음 세션의 USB 점유 충돌을 방지합니다.

---

해제 순서는 다음을 반드시 지킵니다:

1. **`target.dis()`** — UART 채널 (Lite 가 점유 중) 해제
2. **각 scope.dis()** — Husky, Lite 순으로 USB 디바이스 핸들 해제

> ⚠️ **타겟을 먼저 해제하지 않으면 발생하는 문제**
> `target` 객체는 Lite 의 UART 핀을 점유하고 있습니다.
> 이를 닫지 않고 `lite_scope.dis()` 를 먼저 호출하면 차회 실행 시 UART 핀이 점유된 채로 남아 `target` 재생성이 실패할 수 있습니다.

> 💡 **글리치 모듈은 별도 해제가 필요 없다**
> 전압 글리치 모듈은 `husky_scope` 의 내부 서브모듈입니다.
> `husky_scope.dis()` 호출 시 크로우바 트랜지스터가 자동으로 OFF 상태로 복귀하므로, 별도의 `glitch.disable()` 같은 명시적 호출은 필요하지 않습니다.


In [12]:
def disconnect_all_devices(scopes: dict) -> None:
    """전역 ``target``과 ``scopes``의 모든 스코프 연결을 해제한다.

    각 객체의 ``dis()``를 호출한 뒤 입력 딕셔너리를 비우며 반환값은 없다. 개별
    해제 실패는 기록하면서 다음 장치를 계속 처리한 뒤 ``RuntimeError``로
    호출자에게 알린다.
    USB·UART 연결 상태와 입력 딕셔너리를 변경하고 진행 결과를 출력한다.
    """
    failures = []

    # 타겟 객체 먼저 닫아 Lite의 UART 점유를 해제한다.
    try:
        target.dis()
        print("  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료")
    except Exception as e:
        print(f"  [✗] 타겟 보드 연결 해제 실패  └─ {e}")
        failures.append(f'타겟: {e}')

    print("\n장치 연결 해제 중...")
    for name, scope in scopes.items():
        try:
            scope.dis()
            print(f"  [✓] {name} 연결 해제 완료")
        except Exception as e:
            print(f"  [✗] {name} 연결 해제 실패\n      └─ {e}")
            failures.append(f'{name}: {e}')
    scopes.clear()

    if failures:
        raise RuntimeError('연결 해제 실패: ' + '; '.join(failures))

# Trace 수집이 끝나면 다음 세션의 장치 연결과 충돌하지 않도록 포트와 메모리 자원을 반환한다.
disconnect_all_devices(scopes)

  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료

장치 연결 해제 중...
  [✓] ChipWhisperer_Lite 연결 해제 완료
  [✓] ChipWhisperer_Husky 연결 해제 완료


---

## 📝 본 노트북 요약

| 단계 | 핵심 함수 / 명령 | 결과 |
|:----:|:---|:---|
| 1 | 두 `HUSKY_SERIAL_NUMBER` / `LITE_SERIAL_NUMBER` + `cw.scope(sn=...)` | 지정 Lite·Husky 직접 연결 |
| 2 | `cw.target(lite_scope, SimpleSerial2)` | Lite ↔ 타겟 통신 채널 확립 |
| 3 | `make` + `cw.program_target(lite_scope, ...)` | Lite 가 프로그래머로 동작 |
| 4 | `my_fsr_cmd()` + Golden Model 비교 | 통신·연산 정상성 검증 (`Golden_k_XOR_p`) |
| 5 | `vglitch_setup('both')` + `clkgen_src='extclk_aux_io'` + `adc_mul=1` | Husky 가 외부 클럭에 PLL 잠금 + 전압 글리치 무장 |
| 6 | `(ext_offset, offset, width)` 스윕 + 인터랙티브 위젯 모니터링 | 출력 불일치 시행의 Trace·결과·파라미터 수집 |
| 7 | `target.dis()` + 각 `scope.dis()` | 자원 해제 |

### ✅ 본 노트북에서 익혀야 할 핵심 개념

1. **장치 역할 분리** — 통신·프로그래밍·클럭 공급은 Lite, 측정·글리치는 Husky 가 전담하는 다중 장치 협업 구조
2. **시리얼 넘버 기반 명시 연결** — 두 전용 상수를 `cw.scope(sn=...)`에 전달해 장비 역할을 고정
3. **외부 클럭 동기화 + 1 sample = 1 clock** — `extclk_aux_io` 입력 + `freq_ctr` 최빈값 + `adc_mul=1` 의 FIA 친화적 정렬
4. **전압 글리치 모듈의 동시 무장** — `vglitch_setup('both')` + `arm_timing='after_scope'` 로 캡처와 글리치를 한 트리거로 동기 발사
5. **결과 분류와 success_FA 의 선별** — None / infinite-loop / normal / success_FA 의 4-way 분류로 의미 있는 시행만 수집
6. **인터랙티브 모니터링** — `ipywidgets.Output()` 두 개를 활용해 장시간 스윕 중에도 분류 집계와 Trace를 동시에 관찰

---
*Husky 와이어태핑 + 전압 글리치 오류주입 — 응용 노트북 끝*
